In [2]:
# !pip install pandas numpy scanpy scrublet

In [3]:
import sys
sys.path.insert(0, '/home/ubuntu/.local/lib/python3.12/site-packages')

In [4]:
!curl -L -o dropbox_folder_metadata.zip "https://www.dropbox.com/scl/fo/8qvhfjwe6gyevip21eiui/ABQddE-faNklhDdSlSi7wqE?rlkey=ycl9dyhgkt112lm6ygcbrtk2u&dl=1"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     18      0 --:--:-- --:--:-- --:--:--    18
100 13.4M  100 13.4M    0     0  4226k      0  0:00:03  0:00:03 --:--:-- 9110k


In [5]:
!curl -L -o dropbox_folder_data.zip "https://www.dropbox.com/scl/fo/ywteo6su0e6hrw05nfgal/AMBLM6MhDF1FFYmUgTVfVXQ?rlkey=7iy8h29r9xhp24fox6mafo7e1&dl=1"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    17  100    17    0     0     18      0 --:--:-- --:--:-- --:--:--    18
100 3656M  100 3656M    0     0  21.2M      0  0:02:51  0:02:51 --:--:-- 20.2M


In [6]:
!unzip "*.zip"

Archive:  dropbox_folder_metadata.zip
mapname:  conversion of  failed
 extracting: Meta-data_Li2017_Colorectal.tar.gz  
 extracting: Meta-data_Lee2020_Colorectal.tar.gz  
 extracting: Meta-data_Chen2021_Colorectal.tar.gz  
 extracting: Meta-data_Zhang2018_Colorectal.tar.gz  
 extracting: Meta-data_Pelka2021_Colorectal.tar.gz  

Archive:  dropbox_folder_data.zip
mapname:  conversion of  failed
 extracting: Data_Li2017_Colorectal.tar.gz  
 extracting: Data_Lee2020_Colorectal.tar.gz  
 extracting: Data_Chen2021_Colorectal.tar.gz  
 extracting: Data_Zhang2018_Colorectal.tar.gz  
 extracting: Data_Pelka2021_Colorectal.tar.gz  

2 archives had fatal errors.


In [7]:
!for f in *.tar.gz; do tar -xzvf "$f"; done

Data_Chen2021_Colorectal/
Data_Chen2021_Colorectal/Samples.csv
Data_Chen2021_Colorectal/discovery/
Data_Chen2021_Colorectal/discovery/Cells.csv
Data_Chen2021_Colorectal/discovery/Genes.txt
Data_Chen2021_Colorectal/discovery/Exp_data_UMIcounts.mtx
Data_Chen2021_Colorectal/validation/
Data_Chen2021_Colorectal/validation/Cells.csv
Data_Chen2021_Colorectal/validation/Genes.txt
Data_Chen2021_Colorectal/validation/Exp_data_UMIcounts.mtx
Data_Lee2020_Colorectal/
Data_Lee2020_Colorectal/Samples.csv
Data_Lee2020_Colorectal/Cells.csv
Data_Lee2020_Colorectal/Genes.txt
Data_Lee2020_Colorectal/Exp_data_UMIcounts.mtx
Data_Li2017_Colorectal/
Data_Li2017_Colorectal/Exp_data_TPM.mtx
Data_Li2017_Colorectal/Samples.csv
Data_Li2017_Colorectal/Cells.csv
Data_Li2017_Colorectal/Genes.txt
Data_Pelka2021_Colorectal/
Data_Pelka2021_Colorectal/Group4/
Data_Pelka2021_Colorectal/Group4/Cells4.csv
Data_Pelka2021_Colorectal/Group4/Exp_data_UMIcounts4.mtx
Data_Pelka2021_Colorectal/Group2/
Data_Pelka2021_Colorectal/Gr

In [8]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob

In [9]:
# find . -type f | sort

In [10]:
import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [13]:
study_files = {
    "Chen2021_discovery": {
        "matrix": "Data_Chen2021_Colorectal/discovery/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Chen2021_Colorectal/discovery/Genes.txt",
        "cells":  "Data_Chen2021_Colorectal/discovery/Cells.csv",
        "samples": "Data_Chen2021_Colorectal/Samples.csv",
        "units": "UMI"
    },
    "Chen2021_validation": {
        "matrix": "Data_Chen2021_Colorectal/validation/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Chen2021_Colorectal/validation/Genes.txt",
        "cells":  "Data_Chen2021_Colorectal/validation/Cells.csv",
        "samples": "Data_Chen2021_Colorectal/Samples.csv",
        "units": "UMI"
    },
    "Lee2020": {
        "matrix": "Data_Lee2020_Colorectal/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Lee2020_Colorectal/Genes.txt",
        "cells":  "Data_Lee2020_Colorectal/Cells.csv",
        "samples": "Data_Lee2020_Colorectal/Samples.csv",
        "units": "UMI"
    },
    "Li2017": {
        "matrix": "Data_Li2017_Colorectal/Exp_data_TPM.mtx",
        "genes":  "Data_Li2017_Colorectal/Genes.txt",
        "cells":  "Data_Li2017_Colorectal/Cells.csv",
        "samples": "Data_Li2017_Colorectal/Samples.csv",
        "units": "TPM"
    },
    "Pelka2021_Group1": {
        "matrix": "Data_Pelka2021_Colorectal/Group1/Exp_data_UMIcounts1.mtx",
        "genes":  "Data_Pelka2021_Colorectal/Genes.txt",
        "cells":  "Data_Pelka2021_Colorectal/Group1/Cells1.csv",
        "samples": "Data_Pelka2021_Colorectal/Samples.csv",
        "units": "UMI"
    },
    "Pelka2021_Group2": {
        "matrix": "Data_Pelka2021_Colorectal/Group2/Exp_data_UMIcounts2.mtx",
        "genes":  "Data_Pelka2021_Colorectal/Genes.txt",
        "cells":  "Data_Pelka2021_Colorectal/Group2/Cells2.csv",
        "samples": "Data_Pelka2021_Colorectal/Samples.csv",
        "units": "UMI"
    },
    "Pelka2021_Group3": {
        "matrix": "Data_Pelka2021_Colorectal/Group3/Exp_data_UMIcounts3.mtx",
        "genes":  "Data_Pelka2021_Colorectal/Genes.txt",
        "cells":  "Data_Pelka2021_Colorectal/Group3/Cells3.csv",
        "samples": "Data_Pelka2021_Colorectal/Samples.csv",
        "units": "UMI"
    },
    "Pelka2021_Group4": {
        "matrix": "Data_Pelka2021_Colorectal/Group4/Exp_data_UMIcounts4.mtx",
        "genes":  "Data_Pelka2021_Colorectal/Genes.txt",
        "cells":  "Data_Pelka2021_Colorectal/Group4/Cells4.csv",
        "samples": "Data_Pelka2021_Colorectal/Samples.csv",
        "units": "UMI"
    },
    "Zhang2018": {
        "matrix": "Data_Zhang2018_Colorectal/Exp_data_TPM.mtx",
        "genes":  "Data_Zhang2018_Colorectal/Genes.txt",
        "cells":  "Data_Zhang2018_Colorectal/Cells.csv",
        "samples": "Data_Zhang2018_Colorectal/Samples.csv",
        "units": "TPM"
    }
}

In [14]:
study_name = 'Chen2021_discovery'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 65,088 cells × 35,272 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Premalignant' 'Normal' nan]
After gene filter: 64,545 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.82
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 10.0%
Elapsed time: 109.8 seconds
After doublet removal: 64,544 cells
After MT filter: 849 cells
Normalised (UMI) – max value 12.61


In [15]:
study_name = 'Chen2021_validation'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 57,723 cells × 31,412 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Premalignant' 'Normal' nan]
After gene filter: 57,436 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.80
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 1.8%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.3%
Elapsed time: 90.2 seconds
After doublet removal: 57,433 cells
After MT filter: 6,643 cells
Normalised (UMI) – max value 12.52


In [16]:
study_name = 'Lee2020'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 21,657 cells × 22,276 genes
   cell_type: 1030 NaN
   cell_type: 1030 suspicious
Flagged 1030 cells
After metadata drop: 20,627 cells
   Added cancer_type column. Unique values: ['Colorectal Cancer']
After gene filter: 20,627 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.60
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 22.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 19.7 seconds
After doublet removal: 20,619 cells
After MT filter: 20,619 cells
Normalised (UMI) – max value 13.53


In [17]:
study_name = 'Li2017'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 590 cells × 55,017 genes
   cell_type: 121 NaN
   cell_type: 121 suspicious
Flagged 121 cells
After metadata drop: 469 cells
   Added cancer_type column. Unique values: [nan]
After gene filter: 426 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.25
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 2.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 9.1%
Elapsed time: 0.4 seconds
After doublet removal: 425 cells
After MT filter: 117 cells
Normalised (TPM) – max value 12.49


In [18]:
study_name = 'Pelka2021_Group1'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

/tmp/ipykernel_479/3541592848.py:10: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-5', 'SNORD116-6']
  adata.var_names_make_unique()   # added for safety


Loaded: 63,744 cells × 43,113 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Colorectal Cancer']
After gene filter: 61,032 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.51
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 8.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.0%
Elapsed time: 92.2 seconds
After doublet removal: 60,983 cells
After MT filter: 40,380 cells
Normalised (UMI) – max value 13.55


In [19]:
study_name = 'Pelka2021_Group2'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

/tmp/ipykernel_479/2293505245.py:10: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-5', 'SNORD116-6']
  adata.var_names_make_unique()   # added for safety
/tmp/ipykernel_479/2293505245.py:13: DtypeWarning: Columns (15,16,17,18,21,22,23,24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  cells = pd.read_csv(paths["cells"], index_col=0)


Loaded: 117,573 cells × 43,113 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Colorectal Cancer']
After gene filter: 114,906 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.50
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 12.6%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.4%
Elapsed time: 242.8 seconds
After doublet removal: 114,708 cells
After MT filter: 80,103 cells
Normalised (UMI) – max value 13.57


In [20]:
study_name = 'Pelka2021_Group3'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

/tmp/ipykernel_479/4143337052.py:10: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-5', 'SNORD116-6']
  adata.var_names_make_unique()   # added for safety


Loaded: 77,586 cells × 43,113 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Colorectal Cancer']
After gene filter: 75,561 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.51
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 8.2%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.2%
Elapsed time: 122.5 seconds
After doublet removal: 75,487 cells
After MT filter: 51,844 cells
Normalised (UMI) – max value 13.51


In [21]:
study_name = 'Pelka2021_Group4'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

/tmp/ipykernel_479/3770304498.py:10: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`. Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-5', 'SNORD116-6']
  adata.var_names_make_unique()   # added for safety


Loaded: 111,212 cells × 43,113 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Colorectal Cancer']
After gene filter: 99,701 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.65
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 4.4%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.1%
Elapsed time: 192.8 seconds
After doublet removal: 99,653 cells
After MT filter: 46,893 cells
Normalised (UMI) – max value 13.61


In [22]:
study_name = 'Zhang2018'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Colorectal'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 11,138 cells × 22,902 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Colorectal Cancer']
After gene filter: 10,998 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.62
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 3.4%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.5%
Elapsed time: 9.1 seconds
After doublet removal: 10,996 cells
After MT filter: 10,996 cells
Normalised (TPM) – max value 12.65


In [23]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()

Chen2021_discovery  –  849 cells, 35,272 genes
 HTAN.Parent.Data.File.ID: 39 unique values – first 15: HTA11_104_200000101113111, HTA11_104_200000201113111, HTA11_104_200000202113111, HTA11_1391_200000101113111, HTA11_1391_200000101113211, HTA11_1938_200000101113211, HTA11_2112_200000101113211, HTA11_2487_200000101113211, HTA11_2487_200000101113411, HTA11_2487_200000201113111, HTA11_2951_200000101113111, HTA11_2951_200000201113111, HTA11_2951_200000202113111, HTA11_3252_200000201113111, HTA11_3361_200000101113111
 Polyp_Type: HP, NL, SSL, TA, TVA, UNC
 cancer_type: Normal, Premalignant, nan
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling, nan
 cell_subtype: ABS, ASC, CT, EE, GOB, SSC, STM, TAC, TUF
 cell_type: Adenoma_specific_cell, Epithelial, Serrated_specific_cells
 mp_assignment: Cell Cycle, Colon-related, Neuroendocrine, PDAC-related2, PDAC-related3, Respiration, nan
 mp_top: Androgen-prostate, Cell Cycle, Cilia, Colon-related, EMT-like1, Epi1, Epi2, Epi3, Epi4, EpiSen, I

In [25]:
import gc
import anndata as ad
ad.settings.allow_write_nullable_strings = True

INPUT_PATTERN = "*.h5ad"
OUTPUT_FILE = "colorectal.h5ad"

VALID_PREFIXES = [
    'Chen2021_discovery', 'Chen2021_validation',
    'Lee2020', 'Li2017',
    'Pelka2021_Group1', 'Pelka2021_Group2',
    'Pelka2021_Group3', 'Pelka2021_Group4',
    'Zhang2018'
]

all_files = sorted(glob.glob(INPUT_PATTERN))
study_files = [f for f in all_files if any(f.startswith(p) for p in VALID_PREFIXES)]
print(f"Found {len(study_files)} study files (expected 9)\n")

def label_malignant(adata, study_name):
    if 'cell_type' in adata.obs.columns:
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        if ct.str.contains('malignant', na=False).any():
            return ct.str.contains('malignant', na=False), "cell_type='Malignant'"
    if 'source' in adata.obs.columns:
        src = adata.obs['source'].astype(str).str.lower().str.strip()
        if src.str.contains('tumor|tumour', na=False).any():
            return src.str.contains('tumor|tumour', na=False), "source='Tumor'"
    return pd.Series(False, index=adata.obs.index), "all non-malignant"

temp_files = []
total_malig = 0
total_non = 0

for f in study_files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0]
    print(f"Processing {study_name} ({adata.n_obs} cells) …")

    if 'cancer_type' not in adata.obs.columns or adata.obs['cancer_type'].isna().all():
        adata.obs['cancer_type'] = 'Colorectal Cancer'
    else:
        adata.obs['cancer_type'] = adata.obs['cancer_type'].astype(str).fillna('Colorectal Cancer')

    malignant_mask, strategy = label_malignant(adata, study_name)
    adata.obs['is_malignant'] = malignant_mask.astype(int)

    malig_count = malignant_mask.sum()
    non_count = (~malignant_mask).sum()
    total_malig += malig_count
    total_non += non_count
    print(f"   malignant: {malig_count:>6,}   non‑malignant: {non_count:>7,}   ({strategy})")

    essential = ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']
    keep = [c for c in essential if c in adata.obs.columns]
    adata.obs = adata.obs[keep]

    for col in adata.obs.columns:
        try:
            adata.obs[col] = adata.obs[col].astype(str)
        except Exception:
            pass
    adata.obs_names = adata.obs_names.astype(str)

    temp_file = f"__temp_{study_name}.h5ad"
    adata.write_h5ad(temp_file)
    temp_files.append(temp_file)

print(f"\n{'='*60}")
print(f"TOTAL: {total_malig+total_non:>7,} cells  →  malignant: {total_malig:>6,}   non‑malignant: {total_non:>7,}")

print(f"\nMerging {len(temp_files)} studies incrementally …")
adata_all = sc.read_h5ad(temp_files[0])
print(f"   Starting with: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

for i, tf in enumerate(temp_files[1:], 1):
    print(f"   Adding study {i} …")
    ad = sc.read_h5ad(tf)
    adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')
    del ad
    gc.collect()
    print(f"      Combined now: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

print(f"\nFinal merged: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

study_names_original = [sc.read_h5ad(tf).obs['study'].iloc[0] for tf in temp_files]
study_map = {str(i): name for i, name in enumerate(study_names_original)}
adata_all.obs['study'] = adata_all.obs['study'].astype(str).map(study_map)
print(f"Study names restored: {sorted(adata_all.obs['study'].unique())}")

print("\nCancer types:", sorted(adata_all.obs['cancer_type'].astype(str).unique()))
print("Malignant distribution:")
print(adata_all.obs['is_malignant'].value_counts())

for col in adata_all.obs.columns:
    n_nan = adata_all.obs[col].isna().sum()
    if n_nan > 0:
        print(f"{col}: {n_nan} NaN – dropping")
        adata_all = adata_all[~adata_all.obs[col].isna(), :].copy()

for col in adata_all.obs.columns:
    try:
        adata_all.obs[col] = adata_all.obs[col].astype(str)
    except Exception:
        pass
adata_all.obs_names = adata_all.obs_names.astype(str)

adata_all.write_h5ad(OUTPUT_FILE)

for tf in temp_files:
    os.remove(tf)

Found 9 study files (expected 9)

Processing Chen2021_discovery (849 cells) …
   malignant:      0   non‑malignant:     849   (all non-malignant)
Processing Chen2021_validation (6643 cells) …
   malignant:      0   non‑malignant:   6,643   (all non-malignant)
Processing Lee2020 (20619 cells) …
   malignant: 11,072   non‑malignant:   9,547   (cell_type='Malignant')
Processing Li2017 (117 cells) …
   malignant:     55   non‑malignant:      62   (cell_type='Malignant')
Processing Pelka2021_Group1 (40380 cells) …
   malignant:  9,809   non‑malignant:  30,571   (cell_type='Malignant')
Processing Pelka2021_Group2 (80103 cells) …
   malignant: 12,377   non‑malignant:  67,726   (cell_type='Malignant')
Processing Pelka2021_Group3 (51844 cells) …
   malignant: 14,399   non‑malignant:  37,445   (cell_type='Malignant')
Processing Pelka2021_Group4 (46893 cells) …
   malignant:  5,770   non‑malignant:  41,123   (cell_type='Malignant')
Processing Zhang2018 (10996 cells) …
   malignant:  5,048   non‑m

/tmp/ipykernel_479/4112024344.py:79: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')


      Combined now: 7,492 cells × 31,335 genes
   Adding study 2 …


/tmp/ipykernel_479/4112024344.py:79: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')


      Combined now: 28,111 cells × 19,637 genes
   Adding study 3 …


...

      Combined now: 200,555 cells × 17,496 genes
   Adding study 7 …


/tmp/ipykernel_479/4112024344.py:79: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')


      Combined now: 247,448 cells × 17,496 genes
   Adding study 8 …


/tmp/ipykernel_479/4112024344.py:79: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adata_all.concatenate(ad, batch_key='study', join='inner', index_unique='-')


      Combined now: 258,444 cells × 16,839 genes

Final merged: 258,444 cells × 16,839 genes
Study names restored: ['Chen2021_discovery', 'Chen2021_validation']

Cancer types: ['Colorectal Cancer', 'Normal', 'Premalignant', 'nan']
Malignant distribution:
is_malignant
0    199914
1     58530
Name: count, dtype: int64


In [ ]:
# tnr scp 0:/home/ubuntu/colorectal.h5ad ./